<a href="https://colab.research.google.com/github/ShahidMazumdarAI-hub/CHAINS_in_LANGCHAIN/blob/main/Chains__in_Langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***CHAINS in LANGCHAIN***

In [ ]:
OPENAI_API_KEY = "Your api key for openai"
HUGGINGFACEHUB_ACCESS_TOKEN = "your huggingface api key"

In [ ]:
#pipelines can be created using Chains of LANGCHAIN...!!!
#pipeline ki khaasiyat:-> aapke first step ka o/p aapke 2nd step ka i/p ban jata hai and so on
# 3 kinds of chain, at first let's see SEQUENTIAL CHAIN, issey pehle lets see a simple chain
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts  import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers  import StrOutputParser

template1 = PromptTemplate(
    template='Generate 5 interesting facts about {topic}',
    input_variables=['topic']
)

llm = HuggingFaceEndpoint(
    repo_id="HuggingFaceH4/zephyr-7b-beta", #an open source model jo apko by default structured o/p nhi de saktaa
    task="text-generation",
    huggingfacehub_api_token=HUGGINGFACEHUB_ACCESS_TOKEN
)

model = ChatHuggingFace(llm=llm)

parser = StrOutputParser()


chain = template1 | model | parser
result = chain.invoke({'topic':'India'})  #invoke means to trigger
print(result)



# *SEQUENTIAL CHAIN*

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts  import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers  import StrOutputParser

template1 = PromptTemplate(
    template='Fetch me a detailed report on {topic}',
    input_variables=['topic']
)

template2 = PromptTemplate(
    template='Generate a 5 pointer summary from the following text \n {text}',
    input_variables=['text']
)

llm = HuggingFaceEndpoint(
    repo_id="HuggingFaceH4/zephyr-7b-beta", #an open source model jo apko by default structured o/p nhi de saktaa
    task="text-generation",
    huggingfacehub_api_token=HUGGINGFACEHUB_ACCESS_TOKEN
)

model = ChatHuggingFace(llm=llm)

parser = StrOutputParser()

chain = template1 | model | parser | template2 | model | parser
result = chain.invoke({'topic':'Unemployment in India'})
print(result)


# *PARALLEL CHAIN*

In [ ]:
# yaha par user humko text dega like kisi topic ke upar detailed text in the form of a doc
# we will take this doc and generate 2 things
# first:: we will generate notes, then we will generate Quiz and in dono ko combine karke hum user ko dikhaayenge
#for building such apps, we need parallel chain
#lets see how:->
#we will send the doc at first to model1(gpt model) ,this model1 will generate us notes
#at the same time parallely we will take another model(claude), this model will generate us Quiz
#we will take a 3rd model where we will send both notes and Quiz to this 3rd model and this 3rd model will display us the o/p

from langchain_core.runnables import RunnableParallel
from langchain_anthropic import ChatAnthropic
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts  import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers  import StrOutputParser
from langchain_core.runnables import RunnableParallel  #for making parallel chain in L.CHAIN we will need runnable parallel: its a kind of a runnable jiske help se aap parallely multiple chains chalaa sakte ho

llm = HuggingFaceEndpoint(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    task="text-generation",
    huggingfacehub_api_token=HUGGINGFACEHUB_ACCESS_TOKEN
)

model1 = ChatHuggingFace(llm=llm)

llm = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2", # using a diff model in model 2
    task="text-generation",
    huggingfacehub_api_token=HUGGINGFACEHUB_ACCESS_TOKEN
)

model2 = ChatHuggingFace(llm=llm)

template1 = PromptTemplate( #template1 can also be written as prompt1
    template='Generate short and simple notes from the following text \n {text}',
    input_variables=['text']
)

template2 = PromptTemplate(
    template='Generate a 5 short question answers from the following text \n {text}',
    input_variables=['text']
)

template3 = PromptTemplate(
    template='merge the provided notes and quiz into a single document \n notes -> {notes} and quiz -> {quiz}',
    input_variables=['notes','quiz']
)

parser = StrOutputParser()

parallel_chain  = RunnableParallel({
    'notes': template1 | model1 | parser,
    'quiz': template2 | model2 | parser
})

merge_chain = template3 | model1 | parser   #isko aap kisi bhi model mein bhej sakti ho(model1 ya model2) b/c aapko sirf merging karni hai, this one is like sequential chain
chain = parallel_chain | merge_chain        #now parallel chain and merge chain ko milaakar ke aap ek final chain banaaogey

txt = """Cancer is a group of diseases involving abnormal cell growth with the potential to invade or spread to other parts of the body.[2][7] These contrast with benign tumors, which do not spread.[7] Possible signs and symptoms of cancer include a lump, abnormal bleeding, prolonged cough, unexplained weight loss, and a change in bowel movements.[1] While these symptoms may indicate cancer, they can also have other causes.[1] Over 100 types of cancers affect humans.[7][8]

About 33% of deaths from cancer are caused by tobacco and alcohol consumption, obesity, lack of fruit and vegetables in diet and lack of exercise.[2][9][10] Other factors include certain infections, exposure to ionizing radiation, and environmental pollutants.[3] Infection with specific viruses, bacteria, and parasites causes approximately 16–18% of cancers worldwide.[11] These infectious agents include Helicobacter pylori, hepatitis B, hepatitis C, HPV, Epstein–Barr virus, Human T-lymphotropic virus 1, Kaposi's sarcoma-associated herpesvirus and Merkel cell polyomavirus. Human immunodeficiency virus (HIV) does not directly cause cancer, but it causes immune deficiency that can increase the risk of cancer from other infections, sometimes up to several thousandfold (in the case of Kaposi's sarcoma). Importantly, vaccination against the hepatitis B virus and the human papillomavirus have been shown to nearly eliminate the risk of cancers caused by these viruses in persons successfully vaccinated prior to infection.

These environmental factors act, at least partly, by changing the genes of a cell.[12] Typically, many genetic changes are required before cancer develops.[12] Approximately 5–10% of cancers are due to inherited genetic defects.[13] Cancer can be detected by certain signs and symptoms or screening tests.[2] It is then typically further investigated by medical imaging and confirmed by biopsy.[14]

The risk of developing certain cancers can be reduced by not smoking, maintaining a healthy weight, limiting alcohol intake, eating plenty of vegetables, fruits, and whole grains, vaccination against certain infectious diseases, limiting consumption of processed meat and red meat, and limiting exposure to direct sunlight.[15][16] Early detection through screening is useful for cervical and colorectal cancer.[12] The benefits of screening for breast cancer are controversial.[12][17] Cancer is often treated with some combination of radiation therapy, surgery, chemotherapy and targeted therapy.[2][4] More personalized therapies that harness a patient's immune system are emerging in the field of cancer immunotherapy.[18] Palliative care is a medical specialty that delivers advanced pain and symptom management, which may be particularly important in those with advanced disease.[2] The chance of survival depends on the type of cancer and extent of disease at the start of treatment.[12] In children under 15 at diagnosis, the five-year survival rate in the developed world is on average 80%.[19] For cancer in the United States, the average five-year survival rate is 66% for all ages.[5]

In 2015, about 90.5 million people worldwide had cancer.[20] In 2019, annual cancer cases grew by 23.6 million people, and there were 10 million deaths worldwide, representing over the previous decade increases of 26% and 21%, respectively.[6][21]

The most common types of cancer in males are lung cancer, prostate cancer, colorectal cancer, and stomach cancer.[22][23] In females, the most common types are breast cancer, colorectal cancer, lung cancer, and cervical cancer.[12][23] If skin cancer other than melanoma were included in total new cancer cases each year, it would account for around 40% of cases.[24][25] In children, acute lymphoblastic leukemia and brain tumors are most common, except in Africa, where non-Hodgkin lymphoma occurs more often.[19] In 2012, about 165,000 children under 15 years of age were diagnosed with cancer.[22] The risk of cancer increases significantly with age, and many cancers occur more commonly in developed countries.[12] Rates are increasing as more people live to an old age and as lifestyle changes occur in the developing world.[26] The global total economic costs of cancer were estimated at US$1.16 trillion (equivalent to $1.67 trillion in 2024) per year as of 2010.[27]

Duration: 4 minutes and 10 seconds.4:10Subtitles available.CC
Video summary (script)
Etymology and definitions
The word comes from the ancient Greek καρκίνος, meaning 'crab' and 'tumor'. Greek physicians Hippocrates and Galen, among others, noted the similarity of crabs to some tumors with swollen veins. The word was introduced in English in the modern medical sense around 1600.[28]

Cancers comprise a large family of diseases that involve abnormal cell growth with the potential to invade or spread to other parts of the body.[2][7] They form a subset of neoplasms. A neoplasm or tumor is a group of cells that have undergone unregulated growth and will often form a mass or lump, but may be distributed diffusely.[29][30]

All tumor cells show the six hallmarks of cancer. These characteristics are required to produce a malignant tumor. They include:[31]

Cell growth and division absent the proper signals
Continuous growth and division even given contrary signals
Avoidance of programmed cell death
Limitless number of cell divisions
Promoting blood vessel construction
Invasion of tissue and formation of metastases[31]"""

result = chain.invoke({'text':txt})
print(result)

# *CONDITIONAL CHAIN*

In [ ]:
# EK aisa app banaate hai jaha par user hume ek feedback dega about any product, now we need 2 analyze the sentiment of this feedback(+ve or -ve)
#agar feedback +ve aya then i will greet with a thank u, and if -ve then i will apologise and help the customer reach the customer executive to resolve the matter
#user ka feedback will be sent 2 a model, model will analyze the sentiment, agar +ve hai toh yeh feedback ko kisi dusre model ya usi model ke paas bhejke response mangenge accordingly, same for -ve feedback

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers  import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnableLambda #RunnableBranch:-> if else lagaake chains ko execute kar sakte ho is Class ke dwaaraa
                                                                                      #RunnableLambda :-> yeh ek lambda func ko Runnable me convert kar deta hai such that we can use this func as chain
llm = HuggingFaceEndpoint(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    task="text-generation",
    huggingfacehub_api_token=HUGGINGFACEHUB_ACCESS_TOKEN
)

model = ChatHuggingFace(llm=llm)

parser = StrOutputParser()

prompt1 = PromptTemplate(
    template='Classify the sentiment of the following feedback text into positive or negative \n {feedback}',
    input_variables=['feedback']
)
prompt2 = PromptTemplate(
    template='Write an appropiate pleasing response to this positive feedback \n {feedback}',
    input_variables=['feedback']
)
prompt3 = PromptTemplate(
    template='Write an appropiate pleasing response to this negative feedback \n {feedback}',
    input_variables=['feedback']
)

#since we are classifying here, so this can be called as classifier_chain
classifier_chain = prompt1 | model | parser
print(classifier_chain.invoke({'feedback':'this phone is amazing. no software issues'}))




In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnableLambda, RunnablePassthrough #RunnableBranch:-> if else lagaake chains ko execute kar sakte ho is Class ke dwaaraa
from pydantic import BaseModel, Field
from typing import Literal
                                                                                    #RunnableLambda :-> yeh ek lambda func ko Runnable me convert kar deta hai such that we can use this func as chain
llm = HuggingFaceEndpoint(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    task="text-generation",
    huggingfacehub_api_token=HUGGINGFACEHUB_ACCESS_TOKEN
)

model = ChatHuggingFace(llm=llm)
parser = StrOutputParser()

class Feedback(BaseModel):
  sentiment: Literal['positive','negative'] = Field(description='Give the sentiment of the feedback') #sentiment here is an attribute which is a Literal which will be either +ve or -ve

parser2 = PydanticOutputParser(pydantic_object=Feedback)


prompt1 = PromptTemplate(
    template='Classify the sentiment of the following feedback text into positive or negative \n {feedback} \n {format_instruction}',
    input_variables=['feedback'],
    partial_variables={'format_instruction':parser2.get_format_instructions()}
)

classifier_chain = prompt1 | model | parser2 #here i will use parser2 b/c we want pydantic wala parser

print(classifier_chain.invoke({'feedback':'not happy with the smartphone. very slow and lagging!'}))

prompt2 = PromptTemplate(
    template='Write an appropriate pleasing response to this positive feedback: "{feedback}"',
    input_variables=['feedback']
)
prompt3 = PromptTemplate(
    template='Write an appropriate apologetic and helpful response to this negative feedback: "{feedback}"',
    input_variables=['feedback']
)






In [ ]:
#now i will create the branch chain with the help of RunnableBranch for conditional execution
#RunnableBranch me aap multiple tuples bhejte ho, in all tuples u will pass conditions along with(if the conditions are true then which chain u wanna execute) aur last mei agar aapka koi v condition meet nhi ho raha hai toh aap ek default chain execute karte ho .. like if else..if elif elif then at last else
#u can code this part like this below
#branch_chain = RunnableBranch(
   # (condition1, chain),
   # (condition2, chain2),
   # default chain
#)
branch_chain = RunnableBranch(
    (lambda x:x.sentiment == 'positive', prompt2 | model | parser), # here lambda is a function whose i/p is x and this x is the o/p from prompt1,basically if my x is +ve, then send this x as an i/p to prompt2 to execute prompt2
    (lambda x:x.sentiment == 'negative', prompt3 | model | parser),  #basically if my x is -ve, then send this x as an i/p to prompt3 to execute prompt3
    RunnableLambda(lambda x: "Could not find sentiment") #this  line alone lambda x: "Could not find sentiment"  is not a part of chain, so we used RunnableLambda Class here to convert this line in2 a Chain
)

# i will merge both classifier_chain and branch_chain to create my final chain
# then pass to the branching logic.
final_chain = classifier_chain | branch_chain

print(final_chain.invoke({'feedback':'This phone is totally a waste of money. Multiple software issues.'}))

